## Get the German temperture data, using Berlin as a representative example

In [2]:
# Get the German temperture data, using Berlin as a representative example

# Need to already have an time index, from e.g. the electricity demand data

import requests
import pandas as pd

def get_open_meteo_temperature(
    latitude=52.52,
    longitude=13.41,
    start_date="2015-01-01",
    end_date="2020-12-31",
):
    """
    Download daily mean temperature from Open-Meteo archive API.
    Berlin is used by default.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "temperature_2m_mean",
        "timezone": "Europe/Berlin",
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()["daily"]

    temp = pd.DataFrame({
        "date": pd.to_datetime(data["time"]),
        "temperature_2m_mean": data["temperature_2m_mean"],
    })

    temp = temp.set_index("date")
    return temp


## This will fetch the temperature for the time index of your choosing.
## In our case, it comes from 'load' which refers to the electricity load data we have already read in

temp_daily = get_open_meteo_temperature(
    start_date=str(load.index.min().date()),
    end_date=str(load.index.max().date()),
)

NameError: name 'load' is not defined

## Convert to a weekly feature

In [ ]:
# Convert to a weekly feature

temp_weekly = pd.DataFrame(index=weekly.index)

temp_weekly["temp_mean"] = temp_daily["temperature_2m_mean"].resample("W").mean()
temp_weekly["temp_min"] = temp_daily["temperature_2m_mean"].resample("W").min()
temp_weekly["temp_max"] = temp_daily["temperature_2m_mean"].resample("W").max()

# Heating and cooling degree features
base_heat = 15.5
base_cool = 22.0

temp_weekly["heating_degree"] = (
    np.maximum(base_heat - temp_daily["temperature_2m_mean"], 0)
    .resample("W")
    .sum()
)

temp_weekly["cooling_degree"] = (
    np.maximum(temp_daily["temperature_2m_mean"] - base_cool, 0)
    .resample("W")
    .sum()
)

## Merge with e.g. the electricity load data

In [1]:
# merge with e.g. the electricity load data

feature_df = pd.DataFrame({"load_gw": weekly})

feature_df = feature_df.join(weekly_features)
feature_df = feature_df.join(temp_weekly)

feature_df = feature_df.interpolate("time").dropna()

feature_df.head()

NameError: name 'pd' is not defined